In [1]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")

# ── 파일 목록 수집 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")


# ── 병렬 로드 함수 ──
def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    if not instances:
        return None

    duration = data.get("duration", max(inst["end_sec"] for inst in instances))

    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))

        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx,
            "y": gy,
            "w": gw,
            "h": gh,
        })

    return {
        "channel": channel,
        "instances": inst_list,
        "duration": duration,
    }


# ── 병렬 실행 ──
channel_set = set()
samples = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        if result is not None:
            channel_set.add(result["channel"])
            samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

inst_counts = np.array([len(s["instances"]) for s in samples])
print(f"\n✅ 영상: {len(samples):,}개  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")
print(f"📊 인스턴스 수 — mean: {inst_counts.mean():.0f}  p99: {np.percentile(inst_counts, 99):.0f}  max: {inst_counts.max()}")

✅ 임베딩 로드: 1,448,729개  |  dim=1024
📁 파일 수: 66,400개


로드: 100%|██████████| 66400/66400 [00:09<00:00, 6709.96it/s]



✅ 영상: 59,876개  |  채널: 664개
✅ train: 56,556  |  eval: 3,320
📊 인스턴스 수 — mean: 62  p99: 419  max: 4251


In [2]:
# %% 셀 2: Dataset + DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 16
NUM_WORKERS = 8
MASK_RATIO = 1.0


class BLTDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb, mask_ratio=MASK_RATIO):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb
        self.mask_ratio = mask_ratio

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        n = len(s["instances"])
        duration = max(s["duration"], 0.1)

        channel_id = self.channel2id.get(s["channel"], 0)
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)

        text_embs = []
        cond_feats = []
        gt_x, gt_y, gt_w, gt_h = [], [], [], []
        starts, ends = [], []

        for inst in s["instances"]:
            text_embs.append(self.text2emb.get(inst["text"], ZERO_EMB))
            cond_feats.append([
                inst["text_len"] / 200.0,
                inst["start"] / duration,
                inst["end"] / duration,
                (inst["end"] - inst["start"]) / duration,
            ])
            gt_x.append(inst["x"])
            gt_y.append(inst["y"])
            gt_w.append(inst["w"] - 1)
            gt_h.append(inst["h"] - 1)
            starts.append(inst["start"])
            ends.append(inst["end"])

        # 시간 겹침 mask: overlap[i,j] = True이면 동시에 화면에 존재
        starts_arr = np.array(starts)
        ends_arr = np.array(ends)
        overlap = (starts_arr[:, None] < ends_arr[None, :]) & (starts_arr[None, :] < ends_arr[:, None])
        overlap_mask = torch.from_numpy(overlap)  # (N, N) True=겹침=attention 허용

        # 좌표 마스킹
        mask = torch.zeros(n, dtype=torch.bool)
        n_mask = max(1, int(n * self.mask_ratio))
        mask_idx = torch.randperm(n)[:n_mask]
        mask[mask_idx] = True

        return {
            "channel_id": channel_id,
            "channel_emb": channel_emb,
            "text_embs": torch.stack(text_embs),
            "cond_feats": torch.tensor(cond_feats, dtype=torch.float32),
            "gt_x": torch.tensor(gt_x, dtype=torch.long),
            "gt_y": torch.tensor(gt_y, dtype=torch.long),
            "gt_w": torch.tensor(gt_w, dtype=torch.long),
            "gt_h": torch.tensor(gt_h, dtype=torch.long),
            "coord_mask": mask,
            "overlap_mask": overlap_mask,  # (N, N)
            "n": n,
        }


def collate_fn(batch):
    max_n = max(b["n"] for b in batch)
    B = len(batch)

    channel_ids = torch.zeros(B, dtype=torch.long)
    channel_embs = torch.zeros(B, EMB_DIM)
    text_embs = torch.zeros(B, max_n, EMB_DIM)
    cond_feats = torch.zeros(B, max_n, 4)
    gt_x = torch.zeros(B, max_n, dtype=torch.long)
    gt_y = torch.zeros(B, max_n, dtype=torch.long)
    gt_w = torch.zeros(B, max_n, dtype=torch.long)
    gt_h = torch.zeros(B, max_n, dtype=torch.long)
    coord_mask = torch.zeros(B, max_n, dtype=torch.bool)
    seq_mask = torch.zeros(B, max_n, dtype=torch.bool)
    overlap_masks = torch.zeros(B, max_n, max_n, dtype=torch.bool)

    for i, b in enumerate(batch):
        n = b["n"]
        channel_ids[i] = b["channel_id"]
        channel_embs[i] = b["channel_emb"]
        text_embs[i, :n] = b["text_embs"]
        cond_feats[i, :n] = b["cond_feats"]
        gt_x[i, :n] = b["gt_x"]
        gt_y[i, :n] = b["gt_y"]
        gt_w[i, :n] = b["gt_w"]
        gt_h[i, :n] = b["gt_h"]
        coord_mask[i, :n] = b["coord_mask"]
        seq_mask[i, :n] = True
        overlap_masks[i, :n, :n] = b["overlap_mask"]

    return {
        "channel_id": channel_ids,
        "channel_emb": channel_embs,
        "text_embs": text_embs,
        "cond_feats": cond_feats,
        "gt_x": gt_x, "gt_y": gt_y, "gt_w": gt_w, "gt_h": gt_h,
        "coord_mask": coord_mask,
        "seq_mask": seq_mask,
        "overlap_mask": overlap_masks,  # (B, max_n, max_n)
    }


train_ds = BLTDataset(train_samples, channel2id, text2emb, mask_ratio=1.0)
eval_ds = BLTDataset(eval_samples, channel2id, text2emb, mask_ratio=1.0)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

✅ 배치 확인
  channel_id: torch.Size([16])
  channel_emb: torch.Size([16, 1024])
  text_embs: torch.Size([16, 104, 1024])
  cond_feats: torch.Size([16, 104, 4])
  gt_x: torch.Size([16, 104])
  gt_y: torch.Size([16, 104])
  gt_w: torch.Size([16, 104])
  gt_h: torch.Size([16, 104])
  coord_mask: torch.Size([16, 104])
  seq_mask: torch.Size([16, 104])
  overlap_mask: torch.Size([16, 104, 104])


In [3]:
# %% 셀 3: BLT 모델 정의
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
D_FF = 512
DROPOUT = 0.1

N_X = GRID_W   # 80
N_Y = GRID_H   # 80
N_W = GRID_W   # 80
N_H = GRID_H   # 80


class BLTLayoutModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL,
                 n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.n_heads = n_heads

        self.channel_id_emb = nn.Embedding(n_channels, d_model)
        self.channel_text_proj = nn.Linear(emb_dim, d_model)
        self.text_proj = nn.Linear(emb_dim, d_model)

        self.time_proj = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )
        self.textlen_proj = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )

        self.x_emb = nn.Embedding(N_X + 1, d_model // 4)
        self.y_emb = nn.Embedding(N_Y + 1, d_model // 4)
        self.w_emb = nn.Embedding(N_W + 1, d_model // 4)
        self.h_emb = nn.Embedding(N_H + 1, d_model // 4)

        self.MASK_X = N_X
        self.MASK_Y = N_Y
        self.MASK_W = N_W
        self.MASK_H = N_H

        self.coord_proj = nn.Linear(d_model, d_model)

        self.token_norm = nn.LayerNorm(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.head_x = nn.Linear(d_model, N_X)
        self.head_y = nn.Linear(d_model, N_Y)
        self.head_w = nn.Linear(d_model, N_W)
        self.head_h = nn.Linear(d_model, N_H)

        # learnable loss weights
        self.log_var_x = nn.Parameter(torch.zeros(1))
        self.log_var_y = nn.Parameter(torch.zeros(1))
        self.log_var_w = nn.Parameter(torch.zeros(1))
        self.log_var_h = nn.Parameter(torch.zeros(1))

    def encode_coords(self, gt_x, gt_y, gt_w, gt_h, coord_mask):
        x_ids = gt_x.clone()
        y_ids = gt_y.clone()
        w_ids = gt_w.clone()
        h_ids = gt_h.clone()

        x_ids[coord_mask] = self.MASK_X
        y_ids[coord_mask] = self.MASK_Y
        w_ids[coord_mask] = self.MASK_W
        h_ids[coord_mask] = self.MASK_H

        xe = self.x_emb(x_ids)
        ye = self.y_emb(y_ids)
        we = self.w_emb(w_ids)
        he = self.h_emb(h_ids)

        coord = torch.cat([xe, ye, we, he], dim=-1)
        return self.coord_proj(coord)

    def forward(self, channel_id, channel_emb, text_embs, cond_feats,
                gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask, overlap_mask):
        B, N, _ = text_embs.shape

        ch_id = self.channel_id_emb(channel_id).unsqueeze(1).expand(-1, N, -1)
        ch_text = self.channel_text_proj(channel_emb).unsqueeze(1).expand(-1, N, -1)
        text = self.text_proj(text_embs)
        time = self.time_proj(cond_feats[:, :, 1:])
        tlen = self.textlen_proj(cond_feats[:, :, :1])
        coord = self.encode_coords(gt_x, gt_y, gt_w, gt_h, coord_mask)

        tokens = ch_id + ch_text + text + time + tlen + coord
        tokens = self.token_norm(tokens)

        # attention mask: True=차단, False=허용
        attn_mask = ~overlap_mask  # (B, N, N)

        # 자기 자신은 항상 허용
        diag = torch.eye(N, dtype=torch.bool, device=attn_mask.device)
        attn_mask = attn_mask & ~diag.unsqueeze(0)

        # 패딩 행은 전부 허용 (NaN 방지, src_key_padding_mask가 처리)
        padding_rows = ~seq_mask  # (B, N)
        attn_mask[padding_rows] = False

        # multi-head 확장: (B*nhead, N, N)
        attn_mask = attn_mask.unsqueeze(1).expand(-1, self.n_heads, -1, -1)
        attn_mask = attn_mask.reshape(B * self.n_heads, N, N)

        padding_mask = ~seq_mask

        out = self.transformer(tokens, mask=attn_mask, src_key_padding_mask=padding_mask)

        logits_x = self.head_x(out)
        logits_y = self.head_y(out)
        logits_w = self.head_w(out)
        logits_h = self.head_h(out)

        return logits_x, logits_y, logits_w, logits_h


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BLTLayoutModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")

🖥️  Device: cuda
📐 파라미터: 4,060,356


In [ ]:
# %% 셀 4: 학습
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
SAVE_DIR = "./model/8_layout_blt_80x80_overlap"
os.makedirs(SAVE_DIR, exist_ok=True)


def gaussian_soft_label(gt, n_classes, sigma=2.0):
    device = gt.device
    arange = torch.arange(n_classes, device=device).float()
    gt_f = gt.float().unsqueeze(-1)
    label = torch.exp(-0.5 * ((arange - gt_f) / sigma) ** 2)
    label = label / label.sum(dim=-1, keepdim=True)
    return label


def compute_loss(model, logits_x, logits_y, logits_w, logits_h,
                 gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask):
    valid = coord_mask & seq_mask

    if valid.sum() == 0:
        return torch.tensor(0.0, device=logits_x.device, requires_grad=True)

    soft_x = gaussian_soft_label(gt_x[valid], N_X, sigma=2.0)
    soft_y = gaussian_soft_label(gt_y[valid], N_Y, sigma=2.0)
    soft_w = gaussian_soft_label(gt_w[valid], N_W, sigma=2.0)
    soft_h = gaussian_soft_label(gt_h[valid], N_H, sigma=1.0)

    loss_x = F.cross_entropy(logits_x[valid], soft_x)
    loss_y = F.cross_entropy(logits_y[valid], soft_y)
    loss_w = F.cross_entropy(logits_w[valid], soft_w)
    loss_h = F.cross_entropy(logits_h[valid], soft_h)

    total = (loss_x / (2 * torch.exp(model.log_var_x)) + model.log_var_x / 2 +
             loss_y / (2 * torch.exp(model.log_var_y)) + model.log_var_y / 2 +
             loss_w / (2 * torch.exp(model.log_var_w)) + model.log_var_w / 2 +
             loss_h / (2 * torch.exp(model.log_var_h)) + model.log_var_h / 2)

    return total


@torch.no_grad()
def compute_metrics(logits_x, logits_y, logits_w, logits_h,
                    gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask):
    valid = coord_mask & seq_mask

    if valid.sum() == 0:
        return {"mae_x_px": 0, "mae_y_px": 0, "mae_w_px": 0, "mae_h_px": 0,
                "mae_x_grid": 0, "mae_y_grid": 0, "mae_w_grid": 0, "mae_h_grid": 0,
                "acc_x": 0, "acc_y": 0, "acc_w": 0, "acc_h": 0}

    pred_x = logits_x[valid].argmax(dim=-1)
    pred_y = logits_y[valid].argmax(dim=-1)
    pred_w = logits_w[valid].argmax(dim=-1)
    pred_h = logits_h[valid].argmax(dim=-1)

    mae_x_grid = (pred_x - gt_x[valid]).abs().float().mean().item()
    mae_y_grid = (pred_y - gt_y[valid]).abs().float().mean().item()
    mae_w_grid = (pred_w - gt_w[valid]).abs().float().mean().item()
    mae_h_grid = (pred_h - gt_h[valid]).abs().float().mean().item()

    return {
        "mae_x_px": mae_x_grid * CELL_SIZE_X,
        "mae_y_px": mae_y_grid * CELL_SIZE_Y,
        "mae_w_px": mae_w_grid * CELL_SIZE_X,
        "mae_h_px": mae_h_grid * CELL_SIZE_Y,
        "mae_x_grid": mae_x_grid,
        "mae_y_grid": mae_y_grid,
        "mae_w_grid": mae_w_grid,
        "mae_h_grid": mae_h_grid,
        "acc_x": (pred_x == gt_x[valid]).float().mean().item(),
        "acc_y": (pred_y == gt_y[valid]).float().mean().item(),
        "acc_w": (pred_w == gt_w[valid]).float().mean().item(),
        "acc_h": (pred_h == gt_h[valid]).float().mean().item(),
    }


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    # ── train ──
    model.train()
    train_loss_sum, train_batches = 0, 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_id = batch["channel_id"].to(device)
        channel_emb = batch["channel_emb"].to(device)
        text_embs = batch["text_embs"].to(device)
        cond_feats = batch["cond_feats"].to(device)
        gt_x = batch["gt_x"].to(device)
        gt_y = batch["gt_y"].to(device)
        gt_w = batch["gt_w"].to(device)
        gt_h = batch["gt_h"].to(device)
        coord_mask = batch["coord_mask"].to(device)
        seq_mask = batch["seq_mask"].to(device)
        overlap_mask = batch["overlap_mask"].to(device)

        lx, ly, lw, lh = model(channel_id, channel_emb, text_embs, cond_feats,
                                gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask, overlap_mask)
        loss = compute_loss(model, lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    # ── eval ──
    model.eval()
    eval_loss_sum, eval_batches = 0, 0
    all_metrics = {k: [] for k in ["mae_x_px", "mae_y_px", "mae_w_px", "mae_h_px",
                                    "mae_x_grid", "mae_y_grid", "mae_w_grid", "mae_h_grid",
                                    "acc_x", "acc_y", "acc_w", "acc_h"]}

    with torch.no_grad():
        for batch in eval_loader:
            channel_id = batch["channel_id"].to(device)
            channel_emb = batch["channel_emb"].to(device)
            text_embs = batch["text_embs"].to(device)
            cond_feats = batch["cond_feats"].to(device)
            gt_x = batch["gt_x"].to(device)
            gt_y = batch["gt_y"].to(device)
            gt_w = batch["gt_w"].to(device)
            gt_h = batch["gt_h"].to(device)
            coord_mask = batch["coord_mask"].to(device)
            seq_mask = batch["seq_mask"].to(device)
            overlap_mask = batch["overlap_mask"].to(device)

            lx, ly, lw, lh = model(channel_id, channel_emb, text_embs, cond_feats,
                                    gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask, overlap_mask)
            loss = compute_loss(model, lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)
            metrics = compute_metrics(lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1
            for k, v in metrics.items():
                all_metrics[k].append(v)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss = eval_loss_sum / max(eval_batches, 1)
    avg_m = {k: np.mean(v) for k, v in all_metrics.items()}
    lr_now = optimizer.param_groups[0]["lr"]

    wx = 1 / (2 * torch.exp(model.log_var_x)).item()
    wy = 1 / (2 * torch.exp(model.log_var_y)).item()
    ww = 1 / (2 * torch.exp(model.log_var_w)).item()
    wh = 1 / (2 * torch.exp(model.log_var_h)).item()

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"mae: x={avg_m['mae_x_grid']:.2f} y={avg_m['mae_y_grid']:.2f} "
        f"w={avg_m['mae_w_grid']:.2f} h={avg_m['mae_h_grid']:.2f}  "
        f"px: x={avg_m['mae_x_px']:.0f} y={avg_m['mae_y_px']:.0f} "
        f"w={avg_m['mae_w_px']:.0f} h={avg_m['mae_h_px']:.0f}  "
        f"acc: x={avg_m['acc_x']:.3f} y={avg_m['acc_y']:.3f} "
        f"w={avg_m['acc_w']:.3f} h={avg_m['acc_h']:.3f}  "
        f"wt: x={wx:.2f} y={wy:.2f} w={ww:.2f} h={wh:.2f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

[  1/50]  train=6.3377  eval=5.6452  mae: x=13.82 y=20.17 w=13.80 h=1.38  px: x=124 y=323 w=124 h=22  acc: x=0.233 y=0.040 w=0.050 h=0.297  wt: x=0.35 y=0.35 w=0.35 h=0.37  lr=9.99e-05
   💾 best 갱신 (eval_loss=5.6452)


[  2/50]  train=5.2047  eval=4.8611  mae: x=13.12 y=19.02 w=13.39 h=1.33  px: x=118 y=304 w=120 h=21  acc: x=0.203 y=0.058 w=0.056 h=0.325  wt: x=0.26 y=0.26 w=0.26 h=0.29  lr=9.96e-05
   💾 best 갱신 (eval_loss=4.8611)


[  3/50]  train=4.6202  eval=4.4666  mae: x=13.23 y=16.74 w=9.99 h=1.33  px: x=119 y=268 w=90 h=21  acc: x=0.239 y=0.064 w=0.068 h=0.363  wt: x=0.20 y=0.20 w=0.20 h=0.28  lr=9.91e-05
   💾 best 갱신 (eval_loss=4.4666)


[  4/50]  train=4.3530  eval=4.2958  mae: x=12.07 y=16.15 w=9.31 h=1.22  px: x=109 y=258 w=84 h=19  acc: x=0.245 y=0.081 w=0.076 h=0.383  wt: x=0.16 y=0.16 w=0.16 h=0.28  lr=9.84e-05
   💾 best 갱신 (eval_loss=4.2958)


[  5/50]  train=4.2609  eval=4.2694  mae: x=13.03 y=15.77 w=9.75 h=1.25  px: x=117 y=252 w=88 h=20  acc: x=0.236 y=0.085 w=0.075 h=0.395  wt: x=0.14 y=0.13 w=0.14 h=0.28  lr=9.76e-05
   💾 best 갱신 (eval_loss=4.2694)


...

[ 10/50]  train=4.1743  eval=4.1983  mae: x=11.30 y=13.87 w=8.29 h=1.10  px: x=102 y=222 w=75 h=18  acc: x=0.271 y=0.113 w=0.094 h=0.432  wt: x=0.14 y=0.14 w=0.15 h=0.29  lr=9.05e-05
   💾 best 갱신 (eval_loss=4.1983)


[ 11/50]  train=4.1623  eval=4.1973  mae: x=11.04 y=13.52 w=8.38 h=1.10  px: x=99 y=216 w=75 h=18  acc: x=0.273 y=0.117 w=0.095 h=0.430  wt: x=0.14 y=0.14 w=0.15 h=0.30  lr=8.85e-05
   💾 best 갱신 (eval_loss=4.1973)


[ 12/50]  train=4.1529  eval=4.1850  mae: x=10.88 y=13.82 w=8.02 h=1.06  px: x=98 y=221 w=72 h=17  acc: x=0.277 y=0.119 w=0.096 h=0.435  wt: x=0.14 y=0.14 w=0.15 h=0.30  lr=8.64e-05
   💾 best 갱신 (eval_loss=4.1850)


[ 13/50]  train=4.1424  eval=4.1827  mae: x=11.16 y=13.63 w=7.84 h=1.06  px: x=100 y=218 w=71 h=17  acc: x=0.280 y=0.127 w=0.101 h=0.432  wt: x=0.14 y=0.14 w=0.15 h=0.30  lr=8.42e-05
   💾 best 갱신 (eval_loss=4.1827)


[ 14/50]  train=4.1332  eval=4.1819  mae: x=10.98 y=13.06 w=7.91 h=1.06  px: x=99 y=209 w=71 h=17  acc: x=0.281 y=0.123 w=0.102 h=0.444  wt: x=0.14 y=0.14 w=0.15 h=0.30  lr=8.19e-05
   💾 best 갱신 (eval_loss=4.1819)
